In [ ]:
from pathlib import Path
import numpy as np
import rasterio
import pandas as pd
from tqdm import tqdm

RAW_DIR = Path("../data/raw")

BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10"]

In [ ]:
def read_band(city_dir, band):
    path = list(city_dir.glob(f"*.{band}.tif"))[0]
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

records = []

for city_dir in tqdm(sorted(RAW_DIR.iterdir())):
    if not city_dir.is_dir():
        continue

    city = city_dir.name

    for band in BANDS:
        arr = read_band(city_dir, band)

        records.append({
            "city": city,
            "band": band,
            "shape": arr.shape,
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
            "std": float(np.nanstd(arr)),
            "p2": float(np.percentile(arr, 2)),
            "p98": float(np.percentile(arr, 98)),
        })

df = pd.DataFrame(records)
df.head()

In [ ]:
df.groupby("band")[["min", "max", "mean", "std", "p2", "p98"]].mean()